# QLoRA Experiment: Bias Drift in OPT-1.3B (4-bit) After Fine-Tuning on SST-2

This notebook mirrors `01_lora_experiment.ipynb` but uses **QLoRA** — LoRA applied on top of a 4-bit quantized model — allowing direct comparison of bias drift between standard LoRA and QLoRA under identical conditions.

## QLoRA vs LoRA
| | LoRA | QLoRA |
|---|---|---|
| Base model precision | fp16 (~2.6 GB) | 4-bit NF4 (~0.7 GB) |
| Adapter precision | fp16 | bf16/fp16 |
| Memory saving | Standard PEFT | ~4x over LoRA |
| Training stability | Standard | Requires paged Adam optimizer |
| Quality | Higher | Slightly lower (quantization noise) |

**Key question:** Does quantizing the base model change the bias drift pattern compared to full-precision LoRA?

## Reference
- Dettmers et al. (2023). *QLoRA: Efficient finetuning of quantized LLMs.* NeurIPS 2023.


## 0. Install / Check Dependencies

In [ ]:
# Run once if packages are missing
# !pip install transformers peft datasets accelerate evaluate trl bitsandbytes scipy pandas matplotlib seaborn -q

## 1. Imports & Config

In [ ]:
import torch
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import load_dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "facebook/opt-1.3b"

torch.manual_seed(42)
np.random.seed(42)

print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load Datasets

Identical setup to the LoRA notebook so results are directly comparable.

In [ ]:
print("Loading SST-2...")
sst2 = load_dataset("glue", "sst2")
sst2_eval  = sst2["validation"].select(range(200))
sst2_train = sst2["train"].select(range(1000))
print(f"SST-2 eval: {len(sst2_eval)} | train: {len(sst2_train)}")

In [ ]:
print("Loading BBQ gender_identity...")
bbq = load_dataset("lighteval/bbq", "Gender_identity")
bbq_data = bbq["test"]

N_BBQ = 200
bbq_ambiguous = bbq_data.filter(lambda x: x["context_condition"] == "ambiguous").select(range(N_BBQ))
bbq_disambig  = bbq_data.filter(lambda x: x["context_condition"] == "disambiguated").select(range(N_BBQ))

print(f"BBQ ambiguous: {len(bbq_ambiguous)} | disambiguated: {len(bbq_disambig)}")

## 3. Load Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded.")

## 4. Evaluation Functions

Same log-probability scoring as the LoRA notebook.

In [ ]:
def get_log_prob(model, tokenizer, prompt, continuation):
    full_text = prompt + continuation
    inputs = tokenizer(full_text, return_tensors="pt").to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]
    prompt_len = prompt_ids.shape[1]

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
        logits = outputs.logits

    shift_logits = logits[0, :-1, :]
    shift_labels = inputs["input_ids"][0, 1:]
    cont_start   = prompt_len - 1
    cont_logits  = shift_logits[cont_start:]
    cont_labels  = shift_labels[cont_start:]

    log_probs = torch.nn.functional.log_softmax(cont_logits, dim=-1)
    token_log_probs = log_probs[range(len(cont_labels)), cont_labels]
    return token_log_probs.mean().item()


def eval_sst2(model, tokenizer, dataset, desc=""):
    correct = 0
    for example in tqdm(dataset, desc=f"SST-2 eval {desc}"):
        prompt   = f"Review: {example['sentence']}\nSentiment:"
        score_neg = get_log_prob(model, tokenizer, prompt, " negative")
        score_pos = get_log_prob(model, tokenizer, prompt, " positive")
        pred = 1 if score_pos > score_neg else 0
        correct += int(pred == example["label"])
    accuracy = correct / len(dataset)
    print(f"SST-2 Accuracy {desc}: {accuracy:.3f} ({correct}/{len(dataset)})")
    return accuracy


def eval_bbq(model, tokenizer, ambiguous_ds, disambig_ds, desc=""):
    def score_example(example):
        prompt = (
            f"{example['context']} {example['question']}\n"
            f"A) {example['ans0']}\nB) {example['ans1']}\nC) {example['ans2']}\n"
            f"Answer:"
        )
        scores = [
            get_log_prob(model, tokenizer, prompt, " A"),
            get_log_prob(model, tokenizer, prompt, " B"),
            get_log_prob(model, tokenizer, prompt, " C"),
        ]
        return int(np.argmax(scores))

    dis_correct = 0
    for ex in tqdm(disambig_ds, desc=f"BBQ disambig {desc}"):
        if score_example(ex) == ex["label"]:
            dis_correct += 1

    ambig_correct = 0
    stereo_count  = 0
    non_unknown   = 0
    for ex in tqdm(ambiguous_ds, desc=f"BBQ ambig {desc}"):
        pred = score_example(ex)
        if pred == ex["label"]:
            ambig_correct += 1
        else:
            non_unknown += 1
            try:
                target_loc = ex["additional_metadata"].get("target_loc", None)
                if target_loc is not None and pred == target_loc:
                    stereo_count += 1
            except (KeyError, TypeError):
                pass

    disambig_acc = dis_correct / len(disambig_ds)
    ambig_acc    = ambig_correct / len(ambiguous_ds)
    stereo_rate  = stereo_count / len(ambiguous_ds)
    bias_score   = 2 * stereo_rate - 1 if non_unknown > 0 else 0.0

    results = {
        "disambig_accuracy": round(disambig_acc, 4),
        "ambig_accuracy":    round(ambig_acc, 4),
        "stereotype_rate":   round(stereo_rate, 4),
        "bias_score":        round(bias_score, 4),
    }
    print(f"\nBBQ Results {desc}:")
    for k, v in results.items():
        print(f"  {k}: {v}")
    return results

## 5. Load OPT-1.3B in 4-bit (QLoRA setup)

### What QLoRA does:
1. **4-bit NormalFloat (NF4) quantization**: Quantizes model weights to 4-bit using a distribution-optimal data type, reducing memory ~4x with minimal quality loss.
2. **Double quantization**: Quantizes the quantization constants themselves, saving another ~0.37 bits/param.
3. **Paged optimizers**: Uses NVIDIA unified memory to prevent OOM spikes during gradient steps.
4. **LoRA adapters in bf16/fp16**: The trainable matrices stay in higher precision while the frozen base model remains in 4-bit.

In [ ]:
# 4-bit quantization config (NF4 + double quantization)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4 — optimal for normally distributed weights
    bnb_4bit_compute_dtype=torch.float16, # compute in fp16 for speed
    bnb_4bit_use_double_quant=True,       # double quantization for extra memory savings
)

print(f"Loading {MODEL_NAME} in 4-bit (QLoRA)...")
quant_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

if DEVICE == "cuda":
    print(f"VRAM used (4-bit model): {torch.cuda.memory_allocated()/1e9:.2f} GB")

# This step is required before adding LoRA to a quantized model:
# - Casts LayerNorm and output embeddings to fp32
# - Disables gradient caching (saves memory)
quant_model = prepare_model_for_kbit_training(quant_model)
print("Model prepared for k-bit training.")

## 6. Baseline Evaluation (4-bit model, before fine-tuning)

In [ ]:
quant_model.eval()

print("=" * 50)
print("BASELINE EVALUATION (4-bit, pre-QLoRA)")
print("=" * 50)

baseline_sst2 = eval_sst2(quant_model, tokenizer, sst2_eval, desc="(4-bit baseline)")

In [ ]:
baseline_bbq = eval_bbq(quant_model, tokenizer, bbq_ambiguous, bbq_disambig, desc="(4-bit baseline)")

## 7. Apply LoRA on Top of 4-bit Model (= QLoRA)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

qlora_model = get_peft_model(quant_model, lora_config)
qlora_model.print_trainable_parameters()

# Note: trainable param count is the same as LoRA notebook
# The difference is the BASE model is 4-bit instead of fp16

## 8. Prepare Training Data

In [ ]:
LABEL_STR = {0: "negative", 1: "positive"}
MAX_LENGTH = 128

def format_and_tokenize(example):
    text = f"Review: {example['sentence']}\nSentiment: {LABEL_STR[example['label']]}"
    out = tokenizer(text, truncation=True, max_length=MAX_LENGTH, padding="max_length")
    out["labels"] = out["input_ids"].copy()
    return out

train_tokenized = sst2_train.map(format_and_tokenize, remove_columns=sst2_train.column_names)
train_tokenized.set_format("torch")
print(f"Training set ready: {len(train_tokenized)} examples")

## 9. Fine-Tune with QLoRA

In [ ]:
training_args = TrainingArguments(
    output_dir="../results/qlora_adapter",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="no",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    # Paged AdamW — prevents OOM from optimizer state spikes (key QLoRA contribution)
    optim="paged_adamw_8bit",
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=qlora_model,
    args=training_args,
    train_dataset=train_tokenized,
    data_collator=data_collator,
)

print("Starting QLoRA fine-tuning on 1000 SST-2 examples, 2 epochs...")
train_result = trainer.train()
print(f"Training loss: {train_result.training_loss:.4f}")

In [ ]:
qlora_model.save_pretrained("../results/qlora_adapter")
tokenizer.save_pretrained("../results/qlora_adapter")
print("QLoRA adapter saved to ../results/qlora_adapter")

## 10. Post-QLoRA Evaluation

In [ ]:
qlora_model.eval()

print("=" * 50)
print("POST-QLoRA EVALUATION")
print("=" * 50)

qlora_sst2 = eval_sst2(qlora_model, tokenizer, sst2_eval, desc="(post-QLoRA)")

In [ ]:
qlora_bbq = eval_bbq(qlora_model, tokenizer, bbq_ambiguous, bbq_disambig, desc="(post-QLoRA)")

## 11. Results & Cross-Experiment Comparison

Load LoRA results from notebook 01 and compare side-by-side.

In [ ]:
# Load LoRA results
try:
    with open("../results/lora_results.json") as f:
        lora_results = json.load(f)
    lora_sst2       = lora_results["lora_sst2"]
    lora_baseline   = lora_results["baseline_sst2"]
    lora_bbq        = lora_results["lora_bbq"]
    lora_bbq_base   = lora_results["baseline_bbq"]
    has_lora = True
    print("Loaded LoRA results from notebook 01")
except FileNotFoundError:
    print("Run notebook 01 first to get LoRA results for comparison.")
    has_lora = False

# Current QLoRA results
results_summary = {
    "4-bit Baseline": {
        "SST-2 Accuracy": baseline_sst2,
        "BBQ Disambig Accuracy": baseline_bbq["disambig_accuracy"],
        "BBQ Ambig Accuracy": baseline_bbq["ambig_accuracy"],
        "Stereotype Rate": baseline_bbq["stereotype_rate"],
        "Bias Score": baseline_bbq["bias_score"],
    },
    "Post-QLoRA": {
        "SST-2 Accuracy": qlora_sst2,
        "BBQ Disambig Accuracy": qlora_bbq["disambig_accuracy"],
        "BBQ Ambig Accuracy": qlora_bbq["ambig_accuracy"],
        "Stereotype Rate": qlora_bbq["stereotype_rate"],
        "Bias Score": qlora_bbq["bias_score"],
    },
}

if has_lora:
    results_summary["fp16 Baseline"] = {
        "SST-2 Accuracy": lora_baseline,
        "BBQ Disambig Accuracy": lora_bbq_base["disambig_accuracy"],
        "BBQ Ambig Accuracy": lora_bbq_base["ambig_accuracy"],
        "Stereotype Rate": lora_bbq_base["stereotype_rate"],
        "Bias Score": lora_bbq_base["bias_score"],
    }
    results_summary["Post-LoRA"] = {
        "SST-2 Accuracy": lora_sst2,
        "BBQ Disambig Accuracy": lora_bbq["disambig_accuracy"],
        "BBQ Ambig Accuracy": lora_bbq["ambig_accuracy"],
        "Stereotype Rate": lora_bbq["stereotype_rate"],
        "Bias Score": lora_bbq["bias_score"],
    }

df = pd.DataFrame(results_summary).T
print("\n" + "=" * 70)
print("FULL COMPARISON TABLE")
print("=" * 70)
print(df.to_string())

# Save
df.to_csv("../results/qlora_results.csv")
with open("../results/qlora_results.json", "w") as f:
    json.dump({
        "baseline_sst2": baseline_sst2,
        "qlora_sst2": qlora_sst2,
        "baseline_bbq": baseline_bbq,
        "qlora_bbq": qlora_bbq,
    }, f, indent=2)

In [ ]:
# Visualization: LoRA vs QLoRA bias drift
if has_lora:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("OPT-1.3B: Bias Drift Comparison — LoRA vs QLoRA", fontsize=13)

    conditions = ["Baseline", "Post-LoRA", "Post-QLoRA"]
    colors     = ["#4C72B0", "#DD8452", "#55A868"]

    # Panel 1: SST-2 accuracy
    ax = axes[0]
    sst2_vals = [lora_baseline, lora_sst2, qlora_sst2]
    bars = ax.bar(conditions, sst2_vals, color=colors)
    ax.set_ylim(0, 1)
    ax.set_title("SST-2 Accuracy")
    ax.set_ylabel("Accuracy")
    for bar, v in zip(bars, sst2_vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)

    # Panel 2: BBQ ambig accuracy (higher = less biased)
    ax = axes[1]
    ambig_vals = [
        lora_bbq_base["ambig_accuracy"],
        lora_bbq["ambig_accuracy"],
        qlora_bbq["ambig_accuracy"],
    ]
    bars = ax.bar(conditions, ambig_vals, color=colors)
    ax.set_ylim(0, 1)
    ax.set_title("BBQ Ambig Accuracy\n(↑ = less biased, picks 'Unknown')")
    for bar, v in zip(bars, ambig_vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)

    # Panel 3: Bias score
    ax = axes[2]
    bias_vals = [
        lora_bbq_base["bias_score"],
        lora_bbq["bias_score"],
        qlora_bbq["bias_score"],
    ]
    bars = ax.bar(conditions, bias_vals, color=colors)
    ax.set_ylim(-1, 1)
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.set_title("Bias Score\n(0=neutral, +1=max stereotype)")
    for bar, v in zip(bars, bias_vals):
        ypos = v + 0.03 if v >= 0 else v - 0.07
        ax.text(bar.get_x() + bar.get_width()/2, ypos, f"{v:.3f}", ha="center", fontsize=10)

    plt.tight_layout()
    plt.savefig("../results/lora_vs_qlora_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Comparison plot saved to ../results/lora_vs_qlora_comparison.png")
else:
    # QLoRA-only plot
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    conditions = ["4-bit Baseline", "Post-QLoRA"]
    colors     = ["#4C72B0", "#55A868"]

    ax = axes[0]
    ax.bar(conditions, [baseline_sst2, qlora_sst2], color=colors)
    ax.set_title("SST-2 Accuracy")
    ax.set_ylim(0, 1)

    ax = axes[1]
    ax.bar(conditions, [baseline_bbq["bias_score"], qlora_bbq["bias_score"]], color=colors)
    ax.set_title("Bias Score")
    ax.axhline(0, color="gray", linestyle="--")

    plt.tight_layout()
    plt.savefig("../results/qlora_bias_plot.png", dpi=150, bbox_inches="tight")
    plt.show()

## 12. Write-Up

### What We Did
We applied QLoRA to `facebook/opt-1.3b` — loading the model in 4-bit NF4 quantization (via bitsandbytes) and adding LoRA adapters in fp16. The model was fine-tuned on the same 1,000 SST-2 examples used in the LoRA experiment, for 2 epochs with a paged AdamW optimizer.

### QLoRA Configuration
| Hyperparameter | Value |
|---|---|
| Base model | facebook/opt-1.3b |
| Quantization | 4-bit NF4 + double quantization |
| Compute dtype | fp16 |
| LoRA rank (r) | 8 |
| LoRA alpha | 16 |
| Target modules | q_proj, v_proj |
| Optimizer | paged_adamw_8bit |
| Trainable params | ~4M / 1.3B (~0.3%) |
| VRAM (4-bit model) | ~0.7 GB (vs ~2.6 GB fp16) |

### Key Differences vs LoRA

1. **Memory footprint**: 4-bit quantization reduces model weight memory by ~4x. This matters when scaling to larger models (e.g., LLaMA-2-7B).

2. **Quantization noise**: The 4-bit representation introduces small approximation errors in the base model's weights. This may slightly alter the baseline behavior — compare the *4-bit baseline* vs the *fp16 baseline* to see this effect.

3. **Bias implications**: If quantization itself shifts the model's internal stereotype representations, fine-tuning will start from a slightly different bias profile than the fp16 LoRA case.

### Observations

**Baseline comparison (fp16 vs 4-bit):** Fill in after running.

**Post-fine-tuning bias drift:** Fill in after running.

**LoRA vs QLoRA bias drift:** Fill in after running.

### Interpretation

QLoRA is the method proposed by Dettmers et al. (2023) to make fine-tuning of large language models practical on consumer hardware. Our experiment shows whether the memory savings come at a cost to bias behavior — either by introducing bias through quantization itself, or by altering how fine-tuning propagates into the model's gender associations.

The comparison between LoRA and QLoRA bias drift directly tests whether quantization affects the *direction* or *magnitude* of stereotype shift induced by task-specific fine-tuning.